In [3]:
import os
import re
import unicodedata
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    import ftfy  # canonical mojibake fixer; pip install ftfy
except ImportError:
    ftfy = None


def clean_text(s: str) -> str:
    """Repair encoding damage and tidy whitespace, but keep the article's REAL punctuation
    (curly quotes, en/em dashes) so downstream character offsets stay faithful to the source.
    This is the upstream fix that makes fix_mojibake()/_demojibake() elsewhere a safety net."""
    if not s:
        return s
    # 1. repair UTF-8-read-as-Latin-1 damage ("Â£" -> "£", "â€“" -> "–")
    if ftfy is not None:
        s = ftfy.fix_text(s)
    elif any(m in s for m in ("Ã", "Â", "â€")):
        try:
            s = s.encode("latin-1", "ignore").decode("utf-8", "ignore")
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass
    # 2. canonical unicode form + strip invisible junk (NBSP, zero-width, soft hyphen)
    s = unicodedata.normalize("NFC", s)
    s = s.replace(" ", " ").replace("​", "").replace("­", "")
    # 3. tidy whitespace without touching real newlines between paragraphs
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def get_article_text(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    # Pass raw BYTES (not response.text): lets BeautifulSoup sniff the page's declared charset
    # instead of requests defaulting to ISO-8859-1 when the header omits it — that default is the
    # source of the "Â£3.74m" mojibake we were repairing downstream.
    soup = BeautifulSoup(response.content, "html.parser")

    paragraphs = soup.find_all("p")
    article_text = "\n".join(
        p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
    )

    return clean_text(article_text)


def main():
    df = pd.read_csv("articles.csv")

    os.makedirs("articles", exist_ok=True)

    total = len(df)

    for i, row in df.iterrows():
        title = row["title"]
        url = row["url"]
        date = row.get("Date", "")

        print(f"[{i+1}/{total}] Processing: {title}")

        try:
            article_text = get_article_text(url)

            output_file = os.path.join("articles", f"art{i+1}.txt")

            with open(output_file, "w", encoding="utf-8") as f:
                f.write(f"Title: {clean_text(title)}\n")
                f.write(f"URL: {url}\n")
                f.write(f"Date: {date}\n")
                f.write("\n")
                f.write(article_text)

            print(f"    ✓ Saved to {output_file} ({len(article_text)} characters)")

        except Exception as e:
            print(f"    ✗ Failed: {e}")

    print("\nDone!")


if __name__ == "__main__":
    main()

[1/19] Processing: France records 2,025 excess deaths at peak of heatwave as Europe braces for more extreme weather
    ✓ Saved to articles/art1.txt (4571 characters)
[2/19] Processing: Taylor Swift's rumoured wedding celebrations begin with star-studded New York event
    ✓ Saved to articles/art2.txt (7464 characters)
[3/19] Processing: Alibaba to ban employees from using Anthropic's coding tool, source says
    ✗ Failed: 401 Client Error: HTTP Forbidden for url: https://www.reuters.com/world/china/alibaba-ban-claude-code-workplace-over-alleged-backdoor-risks-source-says-2026-07-03/
[4/19] Processing: What is US Open prize money?
    ✓ Saved to articles/art4.txt (2759 characters)
[5/19] Processing: Ebola treatments trial begins in DR Congo
    ✓ Saved to articles/art5.txt (4460 characters)
[6/19] Processing: Plots, love letters and remedies: The medieval secrets being revealed by AI
    ✓ Saved to articles/art6.txt (13798 characters)
[7/19] Processing: Nagelsmann quits as Germany coac

In [4]:
# --- One-time cleanup of the EXISTING dataset/articles/*.txt (no re-scrape) -------------------
# Re-runs clean_text() over the files already on disk so current results are reproducible from
# clean text without re-fetching live pages (which may have changed). Idempotent: files already
# clean are left byte-identical. Run once; safe to re-run.
from pathlib import Path

art_dir = Path("articles")
changed = []
for f in sorted(art_dir.glob("*.txt")):
    original = f.read_text(encoding="utf-8")
    cleaned = clean_text(original)
    if cleaned != original:
        f.write_text(cleaned, encoding="utf-8")
        changed.append(f.name)

print(f"cleaned {len(changed)}/{len(list(art_dir.glob('*.txt')))} files"
      + (":  " + ", ".join(changed) if changed else "  (all already clean)"))


cleaned 0/18 files  (all already clean)


In [5]:
# --- One-time backfill of the Date: header into EXISTING articles/*.txt (no re-scrape) --------
# The old scraper never wrote a Date line, so reference-date resolution (extract_e2e/_reference_date,
# which reads "^Date: dd/mm/yyyy") was getting None. Insert it from articles.csv, mapping row i ->
# art{i+1}.txt, right after the URL line. Idempotent: skips files that already have a header Date.
import pandas as pd
from pathlib import Path

df = pd.read_csv("articles.csv")
art_dir = Path("articles")
added = []
for i, row in df.iterrows():
    f = art_dir / f"art{i+1}.txt"
    if not f.exists():
        continue
    lines = f.read_text(encoding="utf-8").splitlines()
    # locate the URL header line; the Date line goes immediately after it
    url_idx = next((k for k, l in enumerate(lines[:5]) if l.startswith("URL:")), None)
    if url_idx is None:
        print(f"    ! {f.name}: no URL header line, skipped")
        continue
    if url_idx + 1 < len(lines) and lines[url_idx + 1].startswith("Date:"):
        continue  # already backfilled
    lines.insert(url_idx + 1, f"Date: {row.get('Date', '')}")
    f.write_text("\n".join(lines), encoding="utf-8")
    added.append(f.name)

print(f"backfilled Date into {len(added)} file(s)"
      + (":  " + ", ".join(added) if added else "  (all already had a Date header)"))


backfilled Date into 0 file(s)  (all already had a Date header)
